# ML-05 — Feature Vector and Leakage/Privacy Check

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
# ============================================================
# ML-05 — Feature Vector and Leakage / Privacy Check
# Refresh / Content Opportunity Scoring
# ============================================================

from pathlib import Path
import os
import getpass

import numpy as np
import pandas as pd
import duckdb

# ------------------------------------------------------------
# Hugging Face token
# ------------------------------------------------------------

def get_hf_token():
    token = os.environ.get("HF_TOKEN")

    if token:
        return token

    for candidate in (".env", "../.env", "../../.env"):
        p = Path(candidate)

        if p.exists():
            for line in p.read_text().splitlines():
                if line.startswith("HF_TOKEN="):
                    return line.split("=", 1)[1].strip()

    return getpass.getpass(
        "Paste your Hugging Face READ token (hf_...): "
    )


token = get_hf_token().replace("'", "''")

# ------------------------------------------------------------
# DuckDB connection
# ------------------------------------------------------------

con = duckdb.connect()

con.execute("SET enable_progress_bar = false")

con.execute(
    f"""
    CREATE OR REPLACE SECRET hf
    (
        TYPE huggingface,
        TOKEN '{token}'
    )
    """
)

# ------------------------------------------------------------
# Warehouse paths
# ------------------------------------------------------------

REL = "hf://datasets/FlyRank/internship-warehouse"

FACT_FEB = (
    f"{REL}/fact_content_daily_performance/"
    f"month=2026-02/*.parquet"
)

OUT_DIR = Path("work/outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("Connected to FlyRank warehouse.")
print("Feature window: February 2026")
print("Warehouse:", REL)

Paste your Hugging Face READ token (hf_...): ··········
Connected to FlyRank warehouse.
Feature window: February 2026
Warehouse: hf://datasets/FlyRank/internship-warehouse


## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [2]:
# ============================================================
# 1. BUILD THE FEATURE VECTOR
# ============================================================

# February 2026 = feature window
# March 2026 = future/label window
#
# IMPORTANT:
# We do NOT use March data here.
# We also do NOT use trend_direction / trend_pct.

# ------------------------------------------------------------
# 1. Inspect the actual February table
# ------------------------------------------------------------

schema = con.sql(
    f"""
    DESCRIBE
    SELECT *
    FROM read_parquet('{FACT_FEB}')
    """
).df()

print("Available warehouse columns:")
print(
    schema[["column_name", "column_type"]]
    .to_string(index=False)
)

# ------------------------------------------------------------
# 2. Required fields for our safe feature vector
# ------------------------------------------------------------

required_fields = {
    "client_hash_id",
    "content_hash_id",
    "report_date",
    "gsc_impressions",
    "gsc_clicks",
    "gsc_data_available",
}

missing_fields = (
    required_fields
    - set(schema["column_name"])
)

print(
    "\nRequired fields present:",
    not missing_fields
)

if missing_fields:
    raise ValueError(
        f"Missing required fields: "
        f"{sorted(missing_fields)}"
    )

# ------------------------------------------------------------
# 3. Aggregate February data
# ------------------------------------------------------------

features = con.sql(
    f"""
    SELECT
        client_hash_id,
        content_hash_id,

        SUM(
            COALESCE(gsc_impressions, 0)
        ) FILTER (
            WHERE gsc_data_available = TRUE
        ) AS impressions_feb,

        SUM(
            COALESCE(gsc_clicks, 0)
        ) FILTER (
            WHERE gsc_data_available = TRUE
        ) AS clicks_feb,

        COUNT(*) FILTER (
            WHERE gsc_data_available = TRUE
        ) AS measured_days_feb

    FROM read_parquet('{FACT_FEB}')

    WHERE report_date >= DATE '2026-02-01'
      AND report_date <= DATE '2026-02-28'

    GROUP BY
        client_hash_id,
        content_hash_id
    """
).df()

print(
    f"\nRaw feature rows: {len(features):,}"
)

# ------------------------------------------------------------
# 4. Clean numeric fields
# ------------------------------------------------------------

features["impressions_feb"] = pd.to_numeric(
    features["impressions_feb"],
    errors="coerce"
)

features["clicks_feb"] = pd.to_numeric(
    features["clicks_feb"],
    errors="coerce"
)

features["measured_days_feb"] = pd.to_numeric(
    features["measured_days_feb"],
    errors="coerce"
)

# ------------------------------------------------------------
# 5. Keep measured rows only
# ------------------------------------------------------------

features = features[
    features["measured_days_feb"] > 0
].copy()

features["impressions_feb"] = (
    features["impressions_feb"]
    .fillna(0)
)

features["clicks_feb"] = (
    features["clicks_feb"]
    .fillna(0)
)

# ------------------------------------------------------------
# 6. Derived February CTR
# ------------------------------------------------------------

features["ctr_feb"] = np.where(
    features["impressions_feb"] > 0,
    features["clicks_feb"]
    / features["impressions_feb"],
    0.0
)

# ------------------------------------------------------------
# 7. Final feature vector
# ------------------------------------------------------------

feature_columns = [
    "impressions_feb",
    "clicks_feb",
    "ctr_feb",
]

X = features[feature_columns].copy()

print("\nFeature vector shape:")
print(X.shape)

print("\nFeature columns:")
print(X.columns.tolist())

print("\nFeature preview:")
display(X.head())

print(
    "\nPASS — feature vector built "
    "from February observable data."
)

Available warehouse columns:
             column_name column_type
             report_date        DATE
          client_hash_id     VARCHAR
         content_hash_id     VARCHAR
          client_has_gsc     BOOLEAN
          client_has_ga4     BOOLEAN
      gsc_data_available     BOOLEAN
      ga4_data_available     BOOLEAN
         gsc_impressions      BIGINT
              gsc_clicks      BIGINT
        gsc_sum_position      BIGINT
        gsc_avg_position      DOUBLE
           ga4_pageviews      BIGINT
            ga4_sessions      BIGINT
               ga4_users      BIGINT
    ga4_engaged_sessions      BIGINT
ga4_total_engagement_sec      BIGINT
        sessions_organic      BIGINT
         sessions_direct      BIGINT
       sessions_referral      BIGINT
         sessions_social      BIGINT
           sessions_paid      BIGINT
             sessions_ai      BIGINT
              ai_chatgpt      BIGINT
           ai_perplexity      BIGINT
               ai_gemini      BIGINT
         

,impressions_feb,clicks_feb,ctr_feb
18,2.0,0.0,0.0
20,3.0,0.0,0.0
46,1.0,0.0,0.0
47,1.0,0.0,0.0
53,3.0,0.0,0.0



PASS — feature vector built from February observable data.


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

| Feature | Meaning | Missing-value handling | Available before the moment of prediction? |
|---|---|---|---|
| `impressions_feb` | Total Google Search Console impressions observed for the content during February 2026. | Missing GSC impression values are treated as 0 during aggregation. | Yes |
| `clicks_feb` | Total Google Search Console clicks observed for the content during February 2026. | Missing GSC click values are treated as 0 during aggregation. | Yes |
| `ctr_feb` | February click-through rate, calculated as clicks divided by impressions. | CTR is calculated safely; when impressions are 0, CTR is set to 0. | Yes |

### Leakage rationale

These features are constructed only from observable February 2026 data. They do not use future-window outcomes, future labels, or post-prediction information.

Therefore, these features are available at the defined decision point and are suitable as current-window observable signals for the baseline/modeling workflow.

## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

In [5]:
# ============================================================
# 3. THE LEAKAGE HUNT
# ============================================================

print("LEAKAGE HUNT")
print("=" * 70)

# Use the feature vector created in Section 1
if "X" in globals():
    work = X.copy()
else:
    raise NameError(
        "Section 1 variables are missing. "
        "Please run Section 1 before Section 3. "
        "Expected variable 'X'."
    )

print("\nA. Working-frame columns:")
print(list(work.columns))


# ------------------------------------------------------------
# B. Search for suspicious feature names
# ------------------------------------------------------------

danger_keywords = [
    "label",
    "target",
    "future",
    "next",
    "outcome",
    "y_",
    "prediction",
    "predicted"
]

dangerous_scoring_features = [
    col
    for col in work.columns
    if any(
        keyword in col.lower()
        for keyword in danger_keywords
    )
]

print("\nB. Potential label/future-derived features:")
print(dangerous_scoring_features)


# ------------------------------------------------------------
# C. Check whether obvious leakage exists
# ------------------------------------------------------------

if len(dangerous_scoring_features) == 0:

    print("\nPASS - No obvious label-derived or future-window")
    print("features were found in the scoring frame.")

else:

    print("\nREVIEW REQUIRED - Possible leakage features found:")
    print(dangerous_scoring_features)


# ------------------------------------------------------------
# D. Final leakage result
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("LEAKAGE HUNT RESULT")
print("=" * 70)

if len(dangerous_scoring_features) == 0:

    print(
        "PASS - No obvious label-derived or future-window "
        "features were found in the scoring frame."
    )

else:

    print(
        "REVIEW REQUIRED - Possible leakage features were found:"
    )
    print(dangerous_scoring_features)


print(
    "\nThe scoring features use current-window observable "
    "February data only."
)


LEAKAGE HUNT

A. Working-frame columns:
['impressions_feb', 'clicks_feb', 'ctr_feb']

B. Potential label/future-derived features:
[]

PASS - No obvious label-derived or future-window
features were found in the scoring frame.

LEAKAGE HUNT RESULT
PASS - No obvious label-derived or future-window features were found in the scoring frame.

The scoring features use current-window observable February data only.


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

In [6]:
# ============================================================
# 4. WHAT I EXCLUDED AND WHY
# ============================================================

excluded_features = {
    "future_window_metrics":
        "Excluded because they contain information from after the scoring/prediction moment.",

    "label_or_target_fields":
        "Excluded because using the target or a label-derived field as an input would cause data leakage.",

    "future_clicks_or_impressions":
        "Excluded because future performance is not observable at scoring time.",

    "post_period_metrics":
        "Excluded because they would not be available when the decision is made.",

    "client_identifying_fields":
        "Excluded from modeling because public outputs must not expose client-identifying information.",

    "private_queries_or_urls":
        "Excluded to comply with the public-safe data requirements.",

    "credentials_or_private_data":
        "Excluded because credentials and private data must never be used as modeling features."
}

print("WHAT I EXCLUDED AND WHY")
print("=" * 70)

for feature, reason in excluded_features.items():
    print(f"\n{feature}")
    print(f"Reason: {reason}")

print("\n" + "=" * 70)
print("EXCLUSION SUMMARY")
print("=" * 70)

print(
    "The final scoring frame intentionally uses only "
    "observable February performance signals."
)

print(
    "Future-window, label-derived, post-period, and "
    "client-identifying information was excluded."
)

print(
    "\nPASS - Excluded fields were not used as scoring features."
)

WHAT I EXCLUDED AND WHY

future_window_metrics
Reason: Excluded because they contain information from after the scoring/prediction moment.

label_or_target_fields
Reason: Excluded because using the target or a label-derived field as an input would cause data leakage.

future_clicks_or_impressions
Reason: Excluded because future performance is not observable at scoring time.

post_period_metrics
Reason: Excluded because they would not be available when the decision is made.

client_identifying_fields
Reason: Excluded from modeling because public outputs must not expose client-identifying information.

private_queries_or_urls
Reason: Excluded to comply with the public-safe data requirements.

credentials_or_private_data
Reason: Excluded because credentials and private data must never be used as modeling features.

EXCLUSION SUMMARY
The final scoring frame intentionally uses only observable February performance signals.
Future-window, label-derived, post-period, and client-identifying inf

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.